# Projection Accuracy Analysis


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from IPython.display import display, HTML

pd.set_option('display.float_format', '{:.2%}'.format)
pd.set_option('display.max_columns', 20)
sns.set_theme(style='whitegrid', palette='muted')

In [ ]:
FILE_PATH = '../data/processed/projection_dblp_50.csv'   # <-- change this to your file path

df = pd.read_csv(FILE_PATH)

# Normalise the match column: accept both boolean and string variants
if df['match'].dtype == object:
    df['match'] = df['match'].str.strip().str.lower().map({'true': True, 'false': False})
else:
    df['match'] = df['match'].astype(bool)

print(f"Rows: {len(df):,}  |  Columns: {list(df.columns)}")
df.head()

In [ ]:
print("=== Dataset Overview ===")
print(f"  Total predictions : {len(df):,}")
print(f"  Unique models     : {df['model'].nunique()} → {sorted(df['model'].unique())}")
print(f"  Unique columns    : {df['col'].nunique()} → {sorted(df['col'].unique())}")
print(f"  Repetitions       : {sorted(df['repetition'].unique())}")
print(f"  Overall accuracy  : {df['match'].mean():.2%}")
print()
print("Missing values:")
display(df.isnull().sum().rename('nulls').to_frame())

## Accuracy per Model x Column


In [4]:
def plot_model_col(d_frame, m_value):
    df_m = d_frame[d_frame['m']== m_value]

    # Pivot: rows = model, columns = col
    acc_pivot = (
        df_m.groupby(['model', 'col'])['match']
        .agg(accuracy='mean', n_predictions='count', n_correct='sum')
        .reset_index()
    )

    pivot_table = acc_pivot.pivot(index='model', columns='col', values='accuracy')

    model_list = [
    "BSC_300",
    "BSC_512",
    "BSC_1024",
    "BSC_2048",
    "BSC_4096",
    "BSC_8192",
    "HRR_300",
    "HRR_512",
    "HRR_1024",
    "HRR_2048",
    "EmbDI_300",
    "EmbDI_512",
    "SemHDC_FastText_300"
    ]

    # Reorder rows to match model_list, keeping any unexpected models at the end
    ordered_models = [model for model in model_list if model in pivot_table.index]
    remaining_models = [model for model in pivot_table.index if model not in ordered_models]
    pivot_table = pivot_table.reindex(ordered_models + remaining_models)

    # Add row-level aggregates
    pivot_table.insert(0, 'OVERALL', df_m.groupby('model')['match'].mean())
    pivot_table['BEST_COL']  = pivot_table.drop(columns='OVERALL').idxmax(axis=1)
    pivot_table['WORST_COL'] = pivot_table.drop(columns=['OVERALL', 'BEST_COL']).idxmin(axis=1)

    # Colour the numeric cells
    numeric_cols = [c for c in pivot_table.columns if c not in ('BEST_COL', 'WORST_COL')]

    styled = (
        pivot_table.style
        .format({c: '{:.2%}' for c in numeric_cols})
        .background_gradient(cmap='RdYlGn', subset=numeric_cols, vmin=0, vmax=1)
        .set_caption(f'[m={m_value}] Accuracy by Model x Column')
        .set_table_styles([{'selector': 'caption',
                            'props': [('font-size', '14px'), ('font-weight', 'bold'),
                                    ('text-align', 'left'), ('padding-bottom', '6px')]}])
    )
    display(styled)

In [5]:
for m in sorted(df['m'].unique()):
    plot_model_col(df, m)

col,OVERALL,authors,BEST_COL,WORST_COL
model,,,,
BSC_300,100.00%,100.00%,authors,authors
BSC_512,100.00%,100.00%,authors,authors
BSC_1024,100.00%,100.00%,authors,authors
BSC_2048,100.00%,100.00%,authors,authors
BSC_4096,100.00%,100.00%,authors,authors
BSC_8192,100.00%,100.00%,authors,authors
HRR_300,100.00%,100.00%,authors,authors
HRR_512,100.00%,100.00%,authors,authors
HRR_1024,100.00%,100.00%,authors,authors


col,OVERALL,authors,title,BEST_COL,WORST_COL
model,,,,,
BSC_300,100.00%,100.00%,100.00%,authors,authors
BSC_512,100.00%,100.00%,100.00%,authors,authors
BSC_1024,100.00%,100.00%,100.00%,authors,authors
BSC_2048,100.00%,100.00%,100.00%,authors,authors
BSC_4096,100.00%,100.00%,100.00%,authors,authors
BSC_8192,100.00%,100.00%,100.00%,authors,authors
HRR_300,100.00%,100.00%,100.00%,authors,authors
HRR_512,100.00%,100.00%,100.00%,authors,authors
HRR_1024,100.00%,100.00%,100.00%,authors,authors


col,OVERALL,authors,title,venue,BEST_COL,WORST_COL
model,,,,,,
BSC_300,100.00%,100.00%,100.00%,100.00%,authors,authors
BSC_512,100.00%,100.00%,100.00%,100.00%,authors,authors
BSC_1024,100.00%,100.00%,100.00%,100.00%,authors,authors
BSC_2048,100.00%,100.00%,100.00%,100.00%,authors,authors
BSC_4096,100.00%,100.00%,100.00%,100.00%,authors,authors
BSC_8192,100.00%,100.00%,100.00%,100.00%,authors,authors
HRR_300,100.00%,100.00%,100.00%,100.00%,authors,authors
HRR_512,100.00%,100.00%,100.00%,100.00%,authors,authors
HRR_1024,100.00%,100.00%,100.00%,100.00%,authors,authors


col,OVERALL,authors,title,venue,year,BEST_COL,WORST_COL
model,,,,,,,
BSC_300,98.17%,98.67%,96.67%,99.33%,98.00%,venue,title
BSC_512,100.00%,100.00%,100.00%,100.00%,100.00%,authors,authors
BSC_1024,100.00%,100.00%,100.00%,100.00%,100.00%,authors,authors
BSC_2048,100.00%,100.00%,100.00%,100.00%,100.00%,authors,authors
BSC_4096,100.00%,100.00%,100.00%,100.00%,100.00%,authors,authors
BSC_8192,100.00%,100.00%,100.00%,100.00%,100.00%,authors,authors
HRR_300,100.00%,100.00%,100.00%,100.00%,100.00%,authors,authors
HRR_512,100.00%,100.00%,100.00%,100.00%,100.00%,authors,authors
HRR_1024,100.00%,100.00%,100.00%,100.00%,100.00%,authors,authors


## Detailed Stats per Model x Column


In [ ]:
from scipy import stats as scipy_stats

def wilson_ci(successes, n, alpha=0.05):
    """95 % Wilson score confidence interval for a proportion."""
    if n == 0:
        return (np.nan, np.nan)
    z = scipy_stats.norm.ppf(1 - alpha / 2)
    p = successes / n
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denom
    margin = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return (max(0, centre - margin), min(1, centre + margin))

COL_NUM = 4

detail = (
    df[df["m"] == COL_NUM].groupby(['model', 'col'])['match']
      .agg(n='count', correct='sum')
      .reset_index()
)
detail['accuracy']  = detail['correct'] / detail['n']
detail['error_rate'] = 1 - detail['accuracy']
ci = detail.apply(lambda r: wilson_ci(r['correct'], r['n']), axis=1)
detail['ci_low']  = ci.apply(lambda x: x[0])
detail['ci_high'] = ci.apply(lambda x: x[1])
detail['ci_95%']  = detail.apply(
    lambda r: f"[{r['ci_low']:.2%}, {r['ci_high']:.2%}]", axis=1)

fmt_pct = lambda c: '{:.2%}'

display(
    detail[['model', 'col', 'n', 'correct', 'accuracy', 'error_rate', 'ci_95%']]
    .rename(columns={'n': 'predictions', 'correct': 'correct_predictions',
                     'accuracy': 'accuracy_rate', 'error_rate': 'error_rate',
                     'ci_95%': '95% CI (Wilson)'})
    .sort_values(['model', 'col'])
    .style
    .format({'accuracy_rate': '{:.2%}', 'error_rate': '{:.2%}'})
    .background_gradient(cmap='RdYlGn', subset=['accuracy_rate'], vmin=0, vmax=1)
    .set_caption('Detailed Stats with Confidence Intervals')
    .set_table_styles([{'selector': 'caption',
                        'props': [('font-size', '14px'), ('font-weight', 'bold'),
                                  ('text-align', 'left'), ('padding-bottom', '6px')]}])
    .hide(axis='index')
)

## Model-Level Summary


In [ ]:
COL_NUM = 4
model_summary = (
    df[df["m"] == COL_NUM].groupby('model')['match']
      .agg(predictions='count', correct='sum', accuracy='mean')
      .reset_index()
)
model_summary['error_rate'] = 1 - model_summary['accuracy']
model_summary['rank'] = model_summary['accuracy'].rank(ascending=False).astype(int)
model_summary = model_summary.sort_values('rank').reset_index(drop=True)

display(
    model_summary
    .style
    .format({'accuracy': '{:.2%}', 'error_rate': '{:.2%}'})
    .background_gradient(cmap='RdYlGn', subset=['accuracy'], vmin=0, vmax=1)
    .bar(subset=['predictions'], color='#5B9BD5', vmin=0)
    .set_caption(f'Overall Model Summary (ranked for m={COL_NUM})')
    .set_table_styles([{'selector': 'caption',
                        'props': [('font-size', '14px'), ('font-weight', 'bold'),
                                  ('text-align', 'left'), ('padding-bottom', '6px')]}])
    .hide(axis='index')
)

model,predictions,correct,accuracy,error_rate,rank
BSC_2048,750,750,100.00%,0.00%,3
BSC_4096,750,750,100.00%,0.00%,3
BSC_8192,750,750,100.00%,0.00%,3
HRR_1024,750,750,100.00%,0.00%,3
HRR_2048,750,750,100.00%,0.00%,3
BSC_1024,750,739,98.53%,1.47%,6
HRR_512,750,699,93.20%,6.80%,7
EmbDI_300,750,552,73.60%,26.40%,8
EmbDI_512,750,510,68.00%,32.00%,9
BSC_512,750,437,58.27%,41.73%,10


## Column-Level Summary


In [ ]:
col_summary = (
    df.groupby('col')['match']
      .agg(predictions='count', correct='sum', accuracy='mean')
      .reset_index()
      .sort_values('accuracy', ascending=False)
      .reset_index(drop=True)
)
col_summary['difficulty'] = pd.cut(
    col_summary['accuracy'],
    bins=[-0.001, 0.5, 0.75, 0.9, 1.001],
    labels=['Hard (<50%)', 'Medium (50-75%)', 'Easy (75-90%)', 'Very Easy (>90%)']
)

display(
    col_summary
    .style
    .format({'accuracy': '{:.2%}'})
    .background_gradient(cmap='RdYlGn', subset=['accuracy'], vmin=0, vmax=1)
    .set_caption('Column Difficulty Ranking')
    .set_table_styles([{'selector': 'caption',
                        'props': [('font-size', '14px'), ('font-weight', 'bold'),
                                  ('text-align', 'left'), ('padding-bottom', '6px')]}])
    .hide(axis='index')
)

In [ ]:
# --- Heatmap: model × column accuracy ---
heat_data = df.groupby(['model', 'col'])['match'].mean().unstack()

fig, ax = plt.subplots(figsize=(max(6, len(heat_data.columns) * 1.4),
                                max(3, len(heat_data) * 0.9)))
sns.heatmap(
    heat_data, annot=True, fmt='.1%', cmap='RdYlGn',
    vmin=0, vmax=1, linewidths=.5, ax=ax,
    cbar_kws={'format': mtick.PercentFormatter(xmax=1)}
)
ax.set_title('Accuracy Heatmap – Model × Column', fontsize=14, fontweight='bold', pad=10)
ax.set_xlabel('Column', fontsize=11)
ax.set_ylabel('Model', fontsize=11)
plt.tight_layout()
plt.savefig('heatmap_model_column.png', dpi=150, bbox_inches='tight')
plt.show()